# Atelier · De la photo au compte-rendu

Nous savons maintenant lire une photo, comprendre une étiquette globale et
obtenir des objets localisés. Réunissons ces gestes : **une petite application
doit repérer les bus dans une image et rendre un résultat lisible**.

Le programme ne décide pas si une route est sûre. Il annonce ce qu'il a détecté
avec un réglage explicite, conserve les erreurs de lecture et permet une
vérification visuelle. Une **règle de décision** transforme ici les sorties du
modèle en compte-rendu ; cette règle n'est pas apprise par YOLO.

![Le programme vérifie l'image, exécute le détecteur puis compte les boîtes de bus retenues ; zéro boîte ne prouve pas l'absence de bus.](../ressources/illustrations/04_contrat_application.svg)

Nous allons compléter deux petites fonctions, vérifier leurs cas limites,
puis les utiliser sur de vraies images. Le code de lecture et d'inférence
reste fourni : l'exercice porte sur l'interprétation, pas sur la mémorisation d'une API.

Les **figures de référence** sont enregistrées dans ce cours : elles restent
visibles sans exécuter les cellules. Le code produit vos propres résultats ;
si vous changez une image ou un réglage, ils peuvent différer de ces références.


## 1. Définir ce que l'application promet réellement

L'**entrée** sera le chemin d'un fichier image local. Une **détection** sera un
dictionnaire avec une `classe`, un `score` et une `boite` de quatre coordonnées.
Un dictionnaire associe un nom à une valeur : par exemple `detection["score"]`.
C'est le même contenu qu'une ligne du tableau précédent, sous une forme pratique
à transmettre à nos fonctions :

| Dans le tableau du notebook 03 | Dans une détection de l'application |
|---|---|
| Colonne `classe` | `detection["classe"]` |
| Colonne `score` | `detection["score"]` |
| Colonnes `x1`, `y1`, `x2`, `y2` | `detection["boite"]`, liste des quatre coordonnées |

Le **seuil** est le score minimum que notre règle accepte, borne comprise.
Nous utiliserons `0.50` comme valeur de démonstration, **pas comme valeur optimale
validée**. Nous garderons les boîtes dont la classe est exactement `"bus"`.
`"car"` signifie voiture, `"truck"` camion : ces classes ne sont pas des bus.

| Situation | Ce que le compte-rendu doit dire |
|---|---|
| Fichier absent ou illisible | Une erreur de lecture, pas un comptage à zéro. |
| Au moins une boîte de bus retenue | « bus repéré » et le nombre de boîtes. |
| Aucune boîte de bus retenue | « aucun bus détecté — à vérifier ». |

La **NMS**, vue dans le notebook précédent, limite déjà les doublons de boîtes.
Cela ne garantit pas un comptage parfait : un bus peut être manqué, deux objets
peuvent se cacher, une mauvaise boîte peut subsister. On compte des **détections
dans une image**, pas des véhicules uniques suivis à travers une vidéo.


**Figure de référence · visible sans exécution**

![Résultat à obtenir : seul le bus retenu par notre règle est encadré, pas les personnes.](../ressources/illustrations/04_resultat_application.png)

Voici le résultat de référence après les deux fonctions complétées : `bus.jpg`, une boîte bus, seuil `0.50`, message « bus repéré ». Contrairement au notebook 03, les personnes ne sont plus dessinées : notre règle ne les sélectionne pas. Les exercices restent à écrire ci-dessous.


## 2. Reprendre les outils sans dépendre d'un autre notebook ouvert

Le bloc de chemins est le même que dans le premier support. Ce notebook utilise
son propre état Python : exécuter l'autre notebook n'a pas créé ses variables ici.
Le modèle est chargé une seule fois, puis réutilisé pour chaque image.


In [ ]:
from pathlib import Path  # Path manipule un chemin sans écrire les séparateurs Windows à la main.
import cv2               # OpenCV : lire et transformer les pixels.
import numpy as np       # NumPy : manipuler les tableaux de nombres.
import matplotlib.pyplot as plt  # Matplotlib : afficher les images dans le notebook.

# Chercher le dossier du cours depuis ce notebook ou depuis la racine ouverte dans VS Code.
candidats = [Path.cwd(), *Path.cwd().parents]  # Dossier courant, puis ses dossiers parents.
ROOT = next(p for p in candidats if (p / "modeles").is_dir() and (p / "donnees").is_dir())
IMAGES = ROOT / "donnees" / "images"  # Le symbole / assemble les morceaux du chemin.
TRAVAUX = ROOT / "travaux"
TRAVAUX.mkdir(exist_ok=True)  # Créer le dossier de sorties s'il n'existe pas encore.
print("Dossier du cours :", ROOT)


In [ ]:
import os  # Régler les dossiers de travail avant de charger la bibliothèque.
CONFIG = TRAVAUX / "config"
CONFIG.mkdir(parents=True, exist_ok=True)
os.environ["YOLO_CONFIG_DIR"] = str(CONFIG)
os.environ["YOLO_OFFLINE"] = "true"  # Les poids sont fournis localement : aucun téléchargement ici.
os.environ["YOLO_AUTOINSTALL"] = "false"  # Aucune installation depuis le notebook.

from ultralytics import YOLO, settings
settings.update({"sync": False})  # Pas de synchronisation volontaire avec un service distant.
poids = ROOT / "modeles" / "yolo11n.pt"
if not poids.is_file():
    raise FileNotFoundError("Poids absents : suivre INSTALLATION.txt avant cette démonstration.")
detecteur = YOLO(str(poids))  # Le détecteur COCO à 80 classes, pas le classifieur ImageNet.
SEUIL = 0.50  # Choix explicite de l'application, à discuter puis à valider sur d'autres images.


### Une mauvaise entrée n'est pas une absence de bus

La fonction ci-dessous vérifie le chemin puis le décodage. Une **exception**
signale qu'une opération ne peut pas fournir son résultat normal. Nous
intercepterons ces exceptions de lecture dans le traitement de plusieurs fichiers,
sans cacher les autres erreurs de programmation ou d'inférence.


In [ ]:
def lire_image(chemin):
    chemin = Path(chemin)  # Accepter un chemin texte ou un objet Path.
    if not chemin.is_file():
        raise FileNotFoundError(f"Fichier absent : {chemin.name}")
    image = cv2.imread(str(chemin))  # Les pixels restent en BGR pour OpenCV et YOLO.
    if image is None:
        raise ValueError(f"Fichier non décodable comme image : {chemin.name}")
    return image

photo = lire_image(IMAGES / "bus.jpg")
print("Image prête :", photo.shape)


### Transformer la réponse du modèle en lignes faciles à lire

Nous gardons les réglages du notebook 03 : `conf=0.25` et `iou=0.70`.
Le modèle nous fournit ses propositions ; **notre règle** gardera ensuite
seulement les bus dont le score atteint `0.50`. Deux étapes, un seul modèle.
Une proposition supprimée à `0.25` n'est plus disponible pour notre règle.

Une **liste** contient plusieurs éléments. La fonction suivante transforme
chaque boîte en dictionnaire puis l'ajoute à une liste avec `append`.
`float` donne un nombre Python, `int` un entier ; `tolist` convertit
les quatre coordonnées en liste. Les objets du réseau restent visibles.


In [ ]:
def detecter(image_bgr):
    resultats = detecteur.predict(image_bgr, device="cpu", imgsz=640,
                                  conf=0.25, iou=0.70, verbose=False, save=False)
    resultat = resultats[0]  # Une seule image d'entrée, donc un seul résultat.
    lignes = []
    for boite in resultat.boxes:
        identifiant = int(boite.cls[0])
        lignes.append({"classe": resultat.names[identifiant],
                       "score": float(boite.conf[0]),
                       "boite": boite.xyxy[0].tolist()})  # x1, y1, x2, y2 sur l'image d'origine.
    return lignes

detections_photo = detecter(photo)
for detection in detections_photo:
    print(detection["classe"], "— score", round(detection["score"], 3))


## 3. À vous · Ne garder que les bus au seuil choisi

Complétez `retenir_bus(detections, seuil)`. Elle renvoie une **nouvelle liste**
contenant les dictionnaires dont la classe vaut `"bus"` **et** dont le score
est **supérieur ou égal** au seuil. Ne modifiez pas la liste reçue.

Indice : démarrer une liste vide, parcourir les détections avec `for`, tester
les deux conditions avec `and`, puis ajouter un élément avec `append`.
Renvoyez une liste vide `[]` s'il n'y a rien à garder. `None` indique seulement
que l'exercice n'est pas encore complété.


In [ ]:
def retenir_bus(detections, seuil):
    # À compléter : parcourir, vérifier la classe et le score, puis renvoyer la liste.
    return None


In [ ]:
# Données de test inventées pour vérifier la règle, pas prédictions de YOLO.
cas_test = [{"classe": "bus", "score": 0.50, "boite": [0, 0, 10, 10]},
            {"classe": "bus", "score": 0.49, "boite": [20, 0, 30, 10]},
            {"classe": "car", "score": 0.99, "boite": [40, 0, 50, 10]}]
from copy import deepcopy  # Copie indépendante, y compris les listes de coordonnées.
avant_test = deepcopy(cas_test)
selection_test = retenir_bus(cas_test, 0.50)
if selection_test is None:
    print("Exercice 1 à compléter : un seul bus doit être retenu dans ce test.")
else:
    assert selection_test == [cas_test[0]], "Tester la classe ET score >= seuil."
    assert cas_test == avant_test, "Ne pas modifier les détections d'origine."
    assert selection_test is not cas_test, "Renvoyer une nouvelle liste."
    assert retenir_bus([], 0.50) == [], "Une entrée vide doit renvoyer une liste vide."
    assert retenir_bus(cas_test, 1.0) == [], "Aucun score de cet exemple n'atteint 1.0."
    print("Exercice 1 : tests réussis, y compris le score exactement égal au seuil.")


## 4. À vous · Produire un compte-rendu fidèle

Complétez `resumer(nom_image, bus_retenus, seuil)`. Le résultat est un dictionnaire
contenant exactement `image`, `nombre_bus`, `seuil`, `message`.
`nombre_bus` vaut la longueur de la liste, obtenue avec `len`.

Le message vaut `"bus repéré"` si le nombre est positif, sinon
`"aucun bus détecté — à vérifier"`. Le seuil doit figurer dans le résultat :
sans lui, deux comptes-rendus ne sont pas forcément comparables.
Cette fonction résume des boîtes déjà filtrées ; elle ne relance pas le modèle.


In [ ]:
def resumer(nom_image, bus_retenus, seuil):
    # À compléter : construire le dictionnaire attendu avec un message honnête.
    return None


In [ ]:
resume_vide = resumer("exemple.jpg", [], 0.50)
if resume_vide is None:
    print("Exercice 2 à compléter : zéro détection n'est pas une garantie d'absence.")
else:
    assert resume_vide == {"image": "exemple.jpg", "nombre_bus": 0, "seuil": 0.50,
                           "message": "aucun bus détecté — à vérifier"}
    resume_positif = resumer("exemple.jpg", [cas_test[0]], 0.50)
    assert resume_positif == {"image": "exemple.jpg", "nombre_bus": 1, "seuil": 0.50,
                              "message": "bus repéré"}
    resume_deux = resumer("autre.png", [cas_test[0], cas_test[1]], 0.40)
    assert resume_deux == {"image": "autre.png", "nombre_bus": 2, "seuil": 0.40,
                           "message": "bus repéré"}
    print("Exercice 2 : tests réussis pour zéro, une et deux détections.")


## 5. Réunir les étapes et vérifier les boîtes

Nous appliquons maintenant les fonctions à la vraie photo. L'affichage trace
**uniquement les boîtes retenues par notre règle** : sinon, l'image pourrait
montrer plus d'objets que le compte-rendu n'en compte.

La fonction d'affichage utilise une copie : dessiner sur l'image ne modifie pas
les pixels de référence. OpenCV attend des couleurs **BGR** ; ici `(0, 180, 0)`
représente du vert. Les coordonnées sont arrondies en entiers pour le dessin.


In [ ]:
def dessiner_bus(image_bgr, bus_retenus):
    annotee = image_bgr.copy()  # Ne pas dessiner dans l'image d'origine.
    for bus in bus_retenus:
        x1, y1, x2, y2 = [round(v) for v in bus["boite"]]
        cv2.rectangle(annotee, (x1, y1), (x2, y2), (0, 180, 0), 3)
        texte = f"bus {bus['score']:.2f}"  # .2f affiche deux chiffres après la virgule.
        cv2.putText(annotee, texte, (x1, max(25, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 180, 0), 2)
    return annotee


La fonction de dessin est prête. Appliquons maintenant, dans l'ordre :
**sélectionner les bus → résumer → afficher les mêmes bus**.
Tant que les deux exercices ne sont pas complétés, le message d'attente
est normal ; la figure de référence montre le résultat à obtenir.


In [ ]:
bus_photo = retenir_bus(detections_photo, SEUIL)
rapport_photo = None if bus_photo is None else resumer("bus.jpg", bus_photo, SEUIL)
if rapport_photo is None:
    print("Compléter les deux fonctions pour afficher le résultat de votre application.")
else:
    print(rapport_photo)
    plt.figure(figsize=(6, 7))
    plt.imshow(cv2.cvtColor(dessiner_bus(photo, bus_photo), cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()


### Plusieurs fichiers, y compris une mauvaise entrée

L'image `zidane.jpg` est un contre-exemple : deux personnes sur un terrain,
sans bus visible. Le nom `absente.jpg` désigne volontairement un fichier inexistant.
Nous vérifions ainsi deux situations différentes : **rien de recherché n'est
détecté** et **l'image n'a pas pu être analysée**.

`try` tente une opération ; `except` traite seulement les erreurs de lecture
annoncées. `continue` passe au fichier suivant. Commençons par construire
les lignes du compte-rendu et les lire à l’écran, avant de les sauvegarder.


In [ ]:
rapports = []  # Réinitialiser la liste à chaque nouvel essai.
if rapport_photo is None:
    print('Traitement des fichiers en attente des exercices 1 et 2.')
else:
    for nom in ['bus.jpg', 'zidane.jpg', 'absente.jpg']:
        try:
            image = lire_image(IMAGES / nom)
        except (FileNotFoundError, ValueError) as erreur:
            rapports.append({'image': nom, 'nombre_bus': None, 'seuil': SEUIL,
                             'message': str(erreur)})  # None : pas de mesure, pas zéro.
            continue
        bus = retenir_bus(detecter(image), SEUIL)
        rapports.append(resumer(nom, bus, SEUIL))
    for rapport in rapports:
        print(rapport)


### Garder une trace que l'on peut relire

Un **CSV** est un fichier texte organisé en lignes et colonnes, ouvrable
dans un tableur. `DictWriter` utilise les noms du dictionnaire comme colonnes :
`writeheader` écrit les titres, `writerows` écrit les lignes.
Cette étape n'appelle plus le modèle : elle enregistre les résultats déjà lus.

Un nouveau dossier dans `travaux` protège les essais précédents.
La cellule suivante laisse la case `nombre_bus` vide pour une erreur de lecture :
le message explique cette absence de mesure, différente d'un zéro.


In [ ]:
import csv  # Écrire notre tableau dans un fichier.
from datetime import datetime  # Donner un nom distinct à chaque essai.

if rapports:
    dossier_sortie = TRAVAUX / ('analyse_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    dossier_sortie.mkdir()
    with (dossier_sortie / 'rapport.csv').open('w', newline='', encoding='utf-8-sig') as fichier:
        writer = csv.DictWriter(fichier, fieldnames=['image', 'nombre_bus', 'seuil', 'message'])
        writer.writeheader()
        writer.writerows(rapports)
    print('Compte-rendu enregistré dans :', dossier_sortie)
else:
    print('Aucun compte-rendu à enregistrer : compléter les exercices puis relancer les cellules.')


## 6. Une belle démo suffit-elle ? Tester une variation

La **robustesse** décrit la capacité à rester utile quand les conditions changent.
Nous reprenons **la même scène** : une copie est assombrie, une autre est floutée.
Ce sont deux modifications **séparées**, pour comprendre ce qui change.
Le bus reste dans la scène ; il peut devenir plus difficile à reconnaître.

L'assombrissement multiplie chaque intensité par `0.25`. Le flou utilise un
voisinage de `51 × 51` pixels : il atténue davantage les détails que le
`9 × 9` du notebook 01. Le modèle et les seuils ne changent pas.

Ces transformations artificielles ne couvrent pas la diversité d'une caméra
réelle. Réussir les trois essais ne garantit pas les photos futures.


In [ ]:
variantes = {'originale': photo,
             'assombrie': cv2.convertScaleAbs(photo, alpha=0.25, beta=0),
             'floutée': cv2.GaussianBlur(photo, (51, 51), sigmaX=0)}
fig, axes = plt.subplots(1, 3, figsize=(13, 6))
resultats_variantes = []
# items() donne le nom et l'image ; zip associe chaque variante à son panneau.
for axe, (nom, image) in zip(axes, variantes.items()):
    propositions = detecter(image)  # Refaire réellement le calcul sur ces pixels.
    scores_bus = [round(d['score'], 3) for d in propositions if d['classe'] == 'bus']
    bus = retenir_bus(propositions, SEUIL)
    affichee = image if bus is None else dessiner_bus(image, bus)
    axe.imshow(cv2.cvtColor(affichee, cv2.COLOR_BGR2RGB))
    bilan = 'exercice à compléter' if bus is None else f'{len(bus)} boîte(s) retenue(s)'
    axe.set_title(f'{nom} : {bilan}\nScores bus avant notre filtre : {scores_bus}')
    axe.axis('off')
    resultats_variantes.append({'variante': nom, 'scores_bus_avant_filtre': scores_bus})
plt.tight_layout()
plt.show()
for resultat in resultats_variantes:
    print(resultat)


**Figure de référence · visible sans exécution**

![Détection du même bus dans l’original, une copie assombrie et une autre floutée, avec les scores réels.](../ressources/illustrations/04_variantes_reelles.png)

Sur cet essai de référence, les trois variantes gardent une boîte au seuil `0.50`, mais leurs scores diffèrent : environ `0.940`, `0.951` et `0.540`. À `0.60`, la proposition sur l’image floue serait écartée. Le score plus élevé de l’image sombre n’est pas une preuve de meilleure qualité.


### À vous · Interpréter sans promettre trop

1. Le nombre de boîtes retenues change-t-il ? Les scores changent-ils même s'il reste identique ?
2. Si le bus disparaît des résultats de l'image floue, a-t-il disparu de la scène ?
3. Pourquoi trois variantes de la même photo ne forment-elles pas trois scènes indépendantes ?
4. Un seuil plus élevé suffit-il à rendre l'application « sûre » ?
5. Quelles nouvelles photos faudrait-il collecter avant d'utiliser ce système ailleurs ?


Vos observations et vos conclusions : à compléter à partir des sorties réellement affichées.


## 7. Ce que notre chaîne fait, et ce qu'elle ne prouve pas

Nous avons suivi **une photo → des pixels → des hypothèses → une règle → un compte-rendu**.
OpenCV a lu et transformé les pixels. Le classifieur a proposé une catégorie globale.
Le détecteur YOLO a proposé des objets localisés. Notre code a choisi lesquels compter.

Pour évaluer réellement l'application, il faudrait des **annotations de référence** :
des boîtes et des classes vérifiées à la main, sur des scènes variées. On pourrait
comparer prédictions et références pour mesurer les erreurs vues dans le notebook 03.
Il faut garder des images de contrôle final séparées des images utilisées pour
choisir le seuil. Des variantes d'une même photo restent dans le même groupe,
sinon on risque de surestimer la généralisation à des scènes nouvelles.

Une caméra introduirait aussi mouvement, éclairage, cadrage et droits sur les images.
Aucun visage n'est identifié ici et aucun appareil n'est commandé par le résultat.
Un détecteur de bus ne devient pas un détecteur de défauts industriels ou de casques
en changeant simplement le texte affiché : il faudrait d'autres données annotées,
adapter l'apprentissage et valider de nouveau le système.

Pour refaire le parcours : [01 · Comprendre l'image](01_comprendre_et_preparer_une_image.ipynb),
[02 · Classer](02_classer_une_image.ipynb), [03 · Détecter](03_detecter_avec_yolo.ipynb).
Sources des API : [Ultralytics predict](https://docs.ultralytics.com/modes/predict/),
[dessin avec OpenCV](https://docs.opencv.org/4.x/dc/da5/tutorial_py_drawing_functions.html).
